<a href="https://colab.research.google.com/github/prakharyadav-9/sentiment-analysis/blob/bart-model/bart_100p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentencepiece
!pip install transformers
!pip install torch
!pip install rich[jupyter]
!pip install -q sumeval==0.2.2

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import gc
import random
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#import nlpaug.augmenter.word as naw
from sumeval.metrics.rouge import RougeCalculator

import torch
from transformers import AutoTokenizer,AutoModel
import transformers
from transformers import AutoModelForSeq2SeqLM
from transformers import BartTokenizer, BartForConditionalGeneration

print('Pytorch version: %s'  % torch.__version__)

Pytorch version: 2.0.0+cu118


In [ ]:
import nltk
import sklearn

print('The nltk version is {}.'.format(nltk.__version__))
print('The scikit-learn version is {}.'.format(sklearn.__version__))

# The nltk version is 3.0.0.
# The scikit-learn version is 0.15.2.

The nltk version is 3.8.1.
The scikit-learn version is 1.2.2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# from datasets import load_dataset
# train= load_dataset('multi_news', split='train[:60%]')
# test= load_dataset('multi_news', split='test[:60%]')
# val= load_dataset('multi_news', split='validation[:100%]')
# print(len(train))


#our dataset

import json

#f= open("/content/drive/MyDrive/Research Practice/Task1_dev/train/PLOS_train.jsonl")


# #DO NOT RUN
# with open('/content/drive/MyDrive/task1/Task1_dev/train/PLOS_train.jsonl') as f:
#     dataList1 = [json.loads(line) for line in f]


# with open('/content/drive/MyDrive/task1/Task1_dev/train/eLife_train.jsonl') as f:
#     dataList2 = [json.loads(line) for line in f]

In [ ]:
# df1 = pd.DataFrame()
# df2 = pd.DataFrame()
# for data in dataList1:
#   #print(len(data))
#   df_dictionary = pd.DataFrame([data])
#   df1 = pd.concat([df1, df_dictionary], ignore_index=True)

# for data in dataList2:
#   #print(len(data))
#   df_dictionary = pd.DataFrame([data])
#   df2 = pd.concat([df2, df_dictionary], ignore_index=True)
#   # for x in data:
#   #   print(x,data[x])
#   #break

In [ ]:
df1 = pd.read_csv("/content/drive/MyDrive/Research Practice/cleanDatasetPLOS.csv")
df2 = pd.read_csv("/content/drive/MyDrive/Research Practice/cleanDataseteLife.csv")

In [ ]:
#df1

In [ ]:
#df2

In [ ]:
temp = [df1,df2]
df = pd.concat(temp, ignore_index=True)
# df.head()

In [ ]:
df = df.sample(frac = 1)
df=df[:1]

In [ ]:
df.head()

,lay_summary,article,headings,keywords,id,cleanArticle
16083,Echinococcosis is a parasitic disease caused b...,Hydatidosis or echinococcosisis considered a n...,"['Abstract', 'Introduction', 'Materials and Me...",[],journal.pntd.0003934,hydatidosis echinococcosisis considered negle...


In [ ]:
# import pandas as pd
# data = {'text': train['document'],
#         'summary': train['summary']}
# df = pd.DataFrame(data)
# #df = df.sample(14000)

In [ ]:
# Importing libraries
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
import os

from rich.table import Column, Table
from rich import box
from rich.console import Console

# define a rich console logger
console=Console(record=True)

def display_df(df):
  """display dataframe in ASCII format"""

  console=Console()
  table = Table(Column("source_text", justify="center" ), Column("target_text", justify="center"), title="Sample Data",pad_edge=False, box=box.ASCII)

  for i, row in enumerate(df.values.tolist()):
    table.add_row(row[0], row[1])

  console.print(table)

training_logger = Table(Column("Epoch", justify="center" ),
                        Column("Steps", justify="center"),
                        Column("Loss", justify="center"),
                        title="Training Status",pad_edge=False, box=box.ASCII)



In [ ]:
# !pip install -c conda-forge cudatoolkit
!pip install cuda-python

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
# Setting up the device for GPU usage
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

print(device)

cpu


In [ ]:
class YourDataSetClass(Dataset):
  """
  Creating a custom dataset for reading the dataset and
  loading it into the dataloader to pass it to the neural network for finetuning the model

  """

  def __init__(self, dataframe, tokenizer, source_len, target_len, source_text, target_text):
    self.tokenizer = tokenizer
    self.data = dataframe
    self.source_len = source_len
    self.summ_len = target_len
    self.target_text = self.data[target_text]
    self.source_text = self.data[source_text]

  def __len__(self):
    return len(self.target_text)

  def __getitem__(self, index):
    source_text = str(self.source_text[index])
    target_text = str(self.target_text[index])

    #cleaning data so as to ensure data is in string type
    source_text = ' '.join(source_text.split())
    target_text = ' '.join(target_text.split())

    source = self.tokenizer.batch_encode_plus([source_text], max_length= self.source_len, pad_to_max_length=True, truncation=True, padding="max_length", return_tensors='pt')
    target = self.tokenizer.batch_encode_plus([target_text], max_length= self.summ_len, pad_to_max_length=True, truncation=True, padding="max_length", return_tensors='pt')

    source_ids = source['input_ids'].squeeze()
    source_mask = source['attention_mask'].squeeze()
    target_ids = target['input_ids'].squeeze()
    target_mask = target['attention_mask'].squeeze()

    return {
        'source_ids': source_ids.to(dtype=torch.long),
        'source_mask': source_mask.to(dtype=torch.long),
        'target_ids': target_ids.to(dtype=torch.long),
        'target_ids_y': target_ids.to(dtype=torch.long)
    }

In [ ]:
def train(epoch, tokenizer, model, device, loader, optimizer):

  """
  Function to be called for training with the parameters passed from main function

  """

  model.train()
  for _,data in enumerate(loader, 0):
    y = data['target_ids'].to(device, dtype = torch.long)
    y_ids = y[:, :-1].contiguous()
    lm_labels = y[:, 1:].clone().detach()
    lm_labels[y[:, 1:] == tokenizer.pad_token_id] = -100
    ids = data['source_ids'].to(device, dtype = torch.long)
    mask = data['source_mask'].to(device, dtype = torch.long)
    print("above output")
    outputs = model(input_ids = ids, attention_mask = mask, decoder_input_ids=y_ids, labels=lm_labels)
    print("OUTPUT IS: ", outputs)
    loss = outputs[0]

    if _%10==0:
      training_logger.add_row(str(epoch), str(_), str(loss))
      console.print(training_logger)
      torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
            }, './outputs/training_model')



    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [ ]:
def validate(epoch, tokenizer, model, device, loader):

  """
  Function to evaluate model for predictions

  """
  model.eval()
  predictions = []
  actuals = []
  with torch.no_grad():
      for _, data in enumerate(loader, 0):
          y = data['target_ids'].to(device, dtype = torch.long)
          ids = data['source_ids'].to(device, dtype = torch.long)
          mask = data['source_mask'].to(device, dtype = torch.long)

          generated_ids = model.generate(
              input_ids = ids,
              attention_mask = mask,
              max_length=150,
              #num_beams=2,
              repetition_penalty=2.5,
              #length_penalty=1.0,
              #early_stopping=True
              )
          preds = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in generated_ids]
          target = [tokenizer.decode(t, skip_special_tokens=True, clean_up_tokenization_spaces=True)for t in y]
          if _%10==0:
              console.print(f'Completed {_}')

          predictions.extend(preds)
          actuals.extend(target)
  return predictions, actuals

In [ ]:
def BartTrainer(dataframe, source_text, target_text, model_params, output_dir="./outputs/" ):

  """
  Bart trainer

  """

  # Set random seeds and deterministic pytorch for reproducibility
  torch.manual_seed(model_params["SEED"]) # pytorch random seed
  np.random.seed(model_params["SEED"]) # numpy random seed
  torch.backends.cudnn.deterministic = True
  #model = AutoModel.from_pretrained(model_params["MODEL"])
  #tokenizer = AutoTokenizer.from_pretrained(model_params["MODEL"])
  #model = model.to(device)



  # logging
  #console.log(f"""[Model]: Loading {model_params["MODEL"]}...\n""")

  # tokenzier for encoding the text
  ##changing to add distill bart start
  tokenizer = BartTokenizer.from_pretrained(model_params["MODEL"])

  # Defining the model. We are using t5-base model and added a Language model layer on top for generation of Summary.
  # Further this model is sent to device (GPU/TPU) for using the hardware.
  model = BartForConditionalGeneration.from_pretrained(model_params["MODEL"])

  model = model.to(device)


  ##changing to add distill bart end





  # logging
  #console.log(f"[Data]: Reading data...\n")

  # Importing the raw dataset
  dataframe = dataframe[[source_text,target_text]]
  #display_df(dataframe.head(2))


  # Creation of Dataset and Dataloader
  # Defining the train size. So 80% of the data will be used for training and the rest for validation.
######check here how to pass val data fetched directly from dataloader
  train_size = 0.8
  train_dataset=dataframe.sample(frac=train_size,random_state = model_params["SEED"])
  val_dataset=dataframe.drop(train_dataset.index).reset_index(drop=True)
  train_dataset = train_dataset.reset_index(drop=True)

#   console.print(f"FULL Dataset: {dataframe.shape}")
#   console.print(f"TRAIN Dataset: {train_dataset.shape}")
#   console.print(f"TEST Dataset: {val_dataset.shape}\n")


  # Creating the Training and Validation dataset for further creation of Dataloader
  training_set = YourDataSetClass(train_dataset, tokenizer, model_params["MAX_SOURCE_TEXT_LENGTH"], model_params["MAX_TARGET_TEXT_LENGTH"], source_text, target_text)
  val_set = YourDataSetClass(val_dataset, tokenizer, model_params["MAX_SOURCE_TEXT_LENGTH"], model_params["MAX_TARGET_TEXT_LENGTH"], source_text, target_text)


  # Defining the parameters for creation of dataloaders
  train_params = {
      'batch_size': model_params["TRAIN_BATCH_SIZE"],
      'shuffle': True,
      'num_workers': 0
      }


  val_params = {
      'batch_size': model_params["VALID_BATCH_SIZE"],
      'shuffle': False,
      'num_workers': 0
      }


  # Creation of Dataloaders for testing and validation. This will be used down for training and validation stage for the model.
  training_loader = DataLoader(training_set, **train_params)
  val_loader = DataLoader(val_set, **val_params)


  # Defining the optimizer that will be used to tune the weights of the network in the training session.
  optimizer = torch.optim.Adam(params =  model.parameters(), lr=model_params["LEARNING_RATE"])




  # Training loop

  console.log(f'[Initiating Fine Tuning]...\n')

#   if os.path.exists('./outputs/training_model'):
#     checkpoint = torch.load('./outputs/training_model')
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     epoch2 = checkpoint['epoch']
#     epoch=model_params["TRAIN_EPOCHS"]-epoch2
#     loss = checkpoint['loss']

#   else:
#     epoch=model_params["TRAIN_EPOCHS"]

  for epoch1 in range(model_params["TRAIN_EPOCHS"]):
      print("HI")
      train(epoch1, tokenizer, model, device, training_loader, optimizer)

# Training loop ends

  console.log(f"[Saving Model]...\n")
  #Saving the model after training
  path = os.path.join(output_dir, "model_files")
  model.save_pretrained(path)
  tokenizer.save_pretrained(path)


  # evaluating test dataset
  console.log(f"[Initiating Validation]...\n")
  for epoch in range(model_params["VAL_EPOCHS"]):
    predictions, actuals = validate(epoch, tokenizer, model, device, val_loader)
    final_df = pd.DataFrame({'Generated Text':predictions,'Actual Text':actuals})
    final_df.to_csv(os.path.join(output_dir,'predictions.csv'))

  console.save_text(os.path.join(output_dir,'logs.txt'))

  console.log(f"[Validation Completed.]\n")
  console.print(f"""[Model] Model saved @ {os.path.join(output_dir, "model_files")}\n""")
  console.print(f"""[Validation] Generation on Validation data saved @ {os.path.join(output_dir,'predictions.csv')}\n""")
  console.print(f"""[Logs] Logs saved @ {os.path.join(output_dir,'logs.txt')}\n""")


In [ ]:
model_params={
    "MODEL":"facebook/bart-base",#"sshleifer/distilbart-cnn-6-6",#"sshleifer/distilbart-cnn-6-6", #"facebook/bart-base",# model_type: facebook/bart-base
    "TRAIN_BATCH_SIZE":16,          # training batch size
    "VALID_BATCH_SIZE":16,          # validation batch size
    "TRAIN_EPOCHS":1,              # number of training epochs
    "VAL_EPOCHS":2,                # number of validation epochs
    "LEARNING_RATE":2e-5,          # learning rate
    "MAX_SOURCE_TEXT_LENGTH":1024,  # max length of source text
    "MAX_TARGET_TEXT_LENGTH":200,   # max length of target text
    "SEED": 42                     # set seed for reproducibility

}

In [ ]:
df.drop(['article','headings','keywords', 'id'], axis=1, inplace = True)

In [ ]:
df.head()

,lay_summary,cleanArticle
16083,Echinococcosis is a parasitic disease caused b...,hydatidosis echinococcosisis considered negle...


In [ ]:
cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
BartTrainer(dataframe=df ,source_text="cleanArticle", target_text="lay_summary", model_params=model_params, output_dir="/content/drive/MyDrive/outputs")


[08:10:49] [Initiating Fine Tuning]...                                           <ipython-input-22-87a09210bfb9>:92
                                                                                                                   

HI
above output
OUTPUT IS:  Seq2SeqLMOutput(loss=tensor(5.8165, grad_fn=<NllLossBackward0>), logits=tensor([[[ 30.6699,   5.7124,  15.1720,  ...,   6.9007,   6.0522,   5.8643],
         [  4.0615,  -2.5891,   6.1241,  ...,  -1.9882,  -1.6733,  -0.8407],
         [ -1.5169,  -2.2126,   1.5084,  ...,  -2.4060,  -2.6189,  -4.3700],
         ...,
         [ -6.9930,  -4.2519,   1.2285,  ...,  -4.9822,  -5.4639,  -3.2048],
         [-11.3226,  -5.4055,  -0.5778,  ...,  -6.2657,  -5.8682,  -4.0332],
         [ -5.4658,  -4.6937,  -0.5721,  ...,  -5.0424,  -4.7759,  -6.8509]]],
       grad_fn=<AddBackward0>), past_key_values=None, decoder_hidden_states=None, decoder_attentions=None, cross_attentions=None, encoder_last_hidden_state=tensor([[[-0.0361,  0.0142, -0.0006,  ...,  0.0171, -0.0180, -0.0010],
         [ 0.1641,  0.1427,  0.3163,  ..., -0.3609, -0.1402, -0.0213],
         [-0.2231,  0.0633, -0.0645,  ...,  0.1057, -0.0017, -0.1527],
         ...,
         [-0.3305,  0.0544, -0.2307,  .

                      Training Status                       
+----------------------------------------------------------+
|Epoch | Steps |                    Loss                   |
|------+-------+-------------------------------------------|
|  0   |   0   | tensor(5.8165, grad_fn=<NllLossBackward0>)|
+----------------------------------------------------------+

RuntimeError: ignored

In [ ]:
pwd

'/content/drive/MyDrive'

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
output_df = pd.read_csv("./outputs/predictions.csv")
output_df['Generated Text'][9]

'dynamin guanosinetriphosphatase dystrophin related protein plays critical role in mitochondrial fission. This protein is found in many types of cells, but it is not well understood how it interacts with other proteins. For example, when the mitochondria fuse, this protein binds to a specific protein that is responsible for the formation of the cell’s outer mitochondrial membrane. However, it has been suggested that these proteins are also involved in the process of fission, and that they may interact with each other at different times. To investigate this possibility, Chen et al. have now used fluorescent microscopy to study the molecular structure of mitochondria in mice. The experiments showed that after fission, dyst'

In [ ]:
output_df['Actual Text'][9]

'Inside cells, structures called mitochondria supply the energy needed to carry out the processes that sustain life. Mitochondria constantly divide ( a process known as fission ) or fuse together, which helps to keep them in good working condition and well distributed around the cell. Several neurological disorders, including Parkinson’s disease and Alzheimer’s, are associated with problems that affect mitochondrial fission. Many different molecules work together to help mitochondria divide, including a protein called Drp1. A number of Drp1 molecules can associate with each other to form an “oligomer” in the shape of a ring around a mitochondrion. The ring then constricts to split the mitochondrion in two. It is often assumed that Drp1 molecules are recruited to the mitochondria immediately before fission and then form the oligomer ring. However, by using microscopy to track the movement of fluorescently labeled Drp1 molecules in human cells, Ji,'

In [ ]:
from sumeval.metrics.rouge import RougeCalculator

rouge = RougeCalculator(stopwords=True, lang="en")

def rouge_calc(preds, targets):
    rouge_1 = [rouge.rouge_n(summary=preds[i],references=targets[i],n=1) for i in range(len(preds))]
    rouge_2 = [rouge.rouge_n(summary=preds[i],references=targets[i],n=2) for i in range(len(preds))]
    rouge_l = [rouge.rouge_l(summary=preds[i],references=targets[i]) for i in range(len(preds))]

    return {"Rouge_1": np.array(rouge_1).mean(),
            "Rouge_2": np.array(rouge_2).mean(),
            "Rouge_L": np.array(rouge_l).mean()}


In [ ]:
prediction = list(output_df['Generated Text'])
ground_truth = list(output_df['Actual Text'])

rouge_calc(prediction,ground_truth)

{'Rouge_1': 0.21370027650268747,
 'Rouge_2': 0.02722669885745479,
 'Rouge_L': 0.12763397206551755}

In [ ]:
# load the pre-trained best-checkpoint model
from datasets import load_dataset
def inference(model, device, tokenizer, parameters):
#     dataset = load_dataset("multi_news")
#     test = dataset["test"]
    #test = test.remove_columns(["id"])
    test= load_dataset('multi_news', split='test[:5%]')

    sent = test[0]["document"]  # taking an example sentence
    sent = "summarize: " + sent
    sent = " ".join(sent.split())

    source = tokenizer.__call__(
            [sent],
            max_length=model_params["MAX_SOURCE_TEXT_LENGTH"],
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    ids = source["input_ids"]
    mask = source["attention_mask"]

    model.eval()
    with torch.no_grad():
        ids = ids.to(device, dtype = torch.long)
        mask = mask.to(device, dtype = torch.long)

        generated_ids = model.generate(
              input_ids = ids,
              attention_mask = mask,
              max_length=150,
              num_beams=2,
              repetition_penalty=2.5,  # there is a research paper for this
              #length_penalty=1.0,  # > 0 encourages to generate short sentences, < 0 to generate long sentences
              early_stopping=True  # stops beam search when number of beams sentences are generated per batch
              )

        preds = tokenizer.decode(generated_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

        print("Input dialogue is: ", sent)
        print("###############################")
        print("Output summary is: ", preds)
        c=rouge_calc(preds,sent)
        print(c)



cuda =  torch.cuda.is_available()
device = torch.device("cuda") if cuda else torch.device("cpu")

tokenizer = BartTokenizer.from_pretrained("./outputs/model_files", do_lower_case=False)
model = BartForConditionalGeneration.from_pretrained("./outputs/model_files")
model.to(device)
inference(model, device, tokenizer, model_params)

In [ ]:
from IPython.display import display, HTML
import matplotlib as mpl
from matplotlib.colors import Normalize, rgb2hex
import pandas as pd
from IPython.display import HTML
import tensorflow as tf

def get_max_attn(c_atten):
    lst1 = []
    for target,i in enumerate(c_atten):
        lst2 = []
        for ipword in range(512):
            max_head = 0.0
            for layer in range(6):
                max_ = 0
                for head in range(8):
                    if(max_ < c_atten[target][layer][0][head][0][ipword].tolist()):
                        max_ = c_atten[target][layer][0][head][0][ipword].tolist()
                max_head += max_
            avg = max_head/6
            lst2.append(avg)
        lst1.append(lst2)
    return lst1

def predict(model,tokenizer,parameters,sent, device):
    sent = " ".join(sent.split())

    source = tokenizer.__call__(
            [sent],
            max_length=parameters["MAX_SOURCE_TEXT_LENGTH"],
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    ids = source["input_ids"]
    mask = source["attention_mask"]

    model.eval()
    with torch.no_grad():
        ids = ids.to(device, dtype = torch.long)
        mask = mask.to(device, dtype = torch.long)

        generated_ids = model.generate(
              input_ids = ids,
              attention_mask = mask,
              max_length=150,
#               num_beams=2,
              repetition_penalty=2.5,  # there is a research paper for this
              #length_penalty=1.0,  # > 0 encourages to generate short sentences, < 0 to generate long sentences
#               early_stopping=True,  # stops beam search when number of beams sentences are generated per batch
              output_attentions=True,
              return_dict_in_generate=True
              )


        preds = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        print(preds)
    c_atten = generated_ids["cross_attentions"]

    return c_atten, generated_ids, ids

def predict_(model,tokenizer,parameters,sent, device):
    sent = " ".join(sent.split())

    source = tokenizer.__call__(
            [sent],
            max_length=parameters["MAX_SOURCE_TEXT_LENGTH"],
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    ids = source["input_ids"]
    mask = source["attention_mask"]

    model.eval()
    with torch.no_grad():
        ids = ids.to(device, dtype = torch.long)
        mask = mask.to(device, dtype = torch.long)

        generated_ids = model.generate(
              input_ids = ids,
              attention_mask = mask,
              max_length=150,
#               num_beams=2,

              repetition_penalty=2.5,  # there is a research paper for this
              #length_penalty=1.0,  # > 0 encourages to generate short sentences, < 0 to generate long sentences
#               early_stopping=True,  # stops beam search when number of beams sentences are generated per batch
              output_attentions=True,
              return_dict_in_generate=True
              )


        preds = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        print(preds)
        enumerated_preds = tokenizer.convert_ids_to_tokens(generated_ids.sequences[0])
        print("enumerated predictions in token format: ")
        for i,token in enumerate(enumerated_preds):
            print(i,":",enumerated_preds[i])
    c_atten = generated_ids["cross_attentions"]

    return c_atten, generated_ids, ids


def colorize(attrs, cmap='PiYG'):

    cmap_bound = tf.reduce_max(tf.abs(attrs))

    norm = Normalize(vmin=-cmap_bound, vmax=cmap_bound)

    cmap = mpl.cm.get_cmap(cmap)
    colors = list(map(lambda x: rgb2hex(cmap(norm(x))), attrs))

    return colors

def  hlstr(string, color='white'):

    return f"<mark style=background-color:{color}>{string} </mark>"



def color_(max_atten_per_ipword , input_tokens):
    print("*********************")
    input_tokens = [x[1:] for x in input_tokens]
    print("Tokens are : ", input_tokens)

    print("*********************")
    colors = colorize(max_atten_per_ipword)
    colored_input=[]
    display(HTML("".join(list(map(hlstr, input_tokens, colors)))))


def cross_atten(model,tokenizer,parameters,sent, device):

    c_atten, generated_ids, input_ids = predict(model,tokenizer,parameters,sent, device)

    target_input_attn = get_max_attn(c_atten)

    max_atten_per_ipword = []
    for ipword in range(512):
        max_ = 0.0
        for target in range(len(target_input_attn)):
            if(max_ <= target_input_attn[target][ipword]):
                max_ = target_input_attn[target][ipword]
        max_atten_per_ipword.append(max_)
    input_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    input_tokens = [token for token in input_tokens if token != '<pad>']

    color_(max_atten_per_ipword , input_tokens)

def cross_atten_per_word(model,tokenizer,parameters,sent, device):

    c_atten, generated_ids, input_ids = predict_(model,tokenizer,parameters,sent, device)
    tarid = (int)(input("Enter the input id of the target word to be analysed :"))
    target_input_attn = get_max_attn(c_atten)

    input_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    input_tokens = [token for token in input_tokens if token != '<pad>']

    color_(target_input_attn[tarid] , input_tokens)


In [ ]:
tokenizer = BartTokenizer.from_pretrained("./outputs/model_files", do_lower_case=False,add_prefix_space=False)
model = BartForConditionalGeneration.from_pretrained("./outputs/model_files")
model.to(device)

In [ ]:

output_df = pd.read_csv("./outputs/predictions.csv")
cross_atten(model,tokenizer,model_params,output_df["Actual Text"][9], device)

In [ ]:
!tar cvf hundredpercent.tar.gz ./outputs/*
#tar cvzf file.tar.gz *.c

In [ ]:
from IPython.display import FileLink

FileLink(r'hundredpercent.tar.gz')


In [ ]:
!pip install GPUtil
from GPUtil import showUtilization as gpu_usage
gpu_usage()

In [ ]:
torch.cuda.empty_cache()
gc.collect()
gpu_usage()


# Analysis

In [ ]:
tokenizer = BartTokenizer.from_pretrained("./outputs/model_files", do_lower_case=False,add_prefix_space=False)
model = BartForConditionalGeneration.from_pretrained("./outputs/model_files")
model.to(device)

In [ ]:
from sumeval.metrics.rouge import RougeCalculator

rouge = RougeCalculator(stopwords=True, lang="en")

def rouge_calc(preds, targets):
    rouge_1 = rouge.rouge_n(summary=preds,references=targets,n=1)
    rouge_2 = rouge.rouge_n(summary=preds,references=targets,n=2)
    rouge_l = rouge.rouge_l(summary=preds,references=targets)

    return {"Rouge_1": rouge_1,
            "Rouge_2": rouge_2,
            "Rouge_L": rouge_l}


In [ ]:
model_params={
    "MODEL":"facebook/bart-base",#"sshleifer/distilbart-cnn-6-6",#"sshleifer/distilbart-cnn-6-6", #"facebook/bart-base",# model_type: facebook/bart-base
    "TRAIN_BATCH_SIZE":8,          # training batch size
    "VALID_BATCH_SIZE":8,          # validation batch size
    "TRAIN_EPOCHS":1,              # number of training epochs
    "VAL_EPOCHS":2,                # number of validation epochs
    "LEARNING_RATE":2e-5,          # learning rate
    "MAX_SOURCE_TEXT_LENGTH":1024,  # max length of source text
    "MAX_TARGET_TEXT_LENGTH":200,   # max length of target text
    "SEED": 42                     # set seed for reproducibility

}

In [ ]:
from IPython.display import display, HTML
import matplotlib as mpl
from matplotlib.colors import Normalize, rgb2hex
import pandas as pd
from IPython.display import HTML
import tensorflow as tf

def get_max_attn(c_atten):
    lst1 = []
    for target,i in enumerate(c_atten):
        lst2 = []
        for ipword in range(512):
            max_head = 0.0
            for layer in range(6):
                max_ = 0
                for head in range(8):
                    if(max_ < c_atten[target][layer][0][head][0][ipword].tolist()):
                        max_ = c_atten[target][layer][0][head][0][ipword].tolist()
                max_head += max_
            avg = max_head/6
            lst2.append(avg)
        lst1.append(lst2)
    return lst1

def predict(model,tokenizer,parameters,sent, device):
    sent = " ".join(sent.split())

    source = tokenizer.__call__(
            [sent],
            max_length=parameters["MAX_SOURCE_TEXT_LENGTH"],
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    ids = source["input_ids"]
    mask = source["attention_mask"]

    model.eval()
    with torch.no_grad():
        ids = ids.to(device, dtype = torch.long)
        mask = mask.to(device, dtype = torch.long)

        generated_ids = model.generate(
              input_ids = ids,
              attention_mask = mask,
              max_length=150,
#               num_beams=2,
              repetition_penalty=2.5,  # there is a research paper for this
              #length_penalty=1.0,  # > 0 encourages to generate short sentences, < 0 to generate long sentences
#               early_stopping=True,  # stops beam search when number of beams sentences are generated per batch
              output_attentions=True,
              return_dict_in_generate=True
              )


        preds = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        print("May be I'm the issue: \n",preds)
    c_atten = generated_ids["cross_attentions"]

    return c_atten, generated_ids, ids

def predict_(model,tokenizer,parameters,sent, device):
    sent = " ".join(sent.split())

    source = tokenizer.__call__(
            [sent],
            max_length=parameters["MAX_SOURCE_TEXT_LENGTH"],
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    ids = source["input_ids"]
    mask = source["attention_mask"]

    model.eval()
    with torch.no_grad():
        ids = ids.to(device, dtype = torch.long)
        mask = mask.to(device, dtype = torch.long)

        generated_ids = model.generate(
              input_ids = ids,
              attention_mask = mask,
              max_length=150,
#               num_beams=2,
              repetition_penalty=2.5,  # there is a research paper for this
              #length_penalty=1.0,  # > 0 encourages to generate short sentences, < 0 to generate long sentences
#               early_stopping=True,  # stops beam search when number of beams sentences are generated per batch
              output_attentions=True,
              return_dict_in_generate=True
              )


        preds = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
#         print(preds)
        enumerated_preds = tokenizer.convert_ids_to_tokens(generated_ids.sequences[0])
        print("enumerated predictions in token format: ")
        for i,token in enumerate(enumerated_preds):
            print(i,":",enumerated_preds[i])
    c_atten = generated_ids["cross_attentions"]

    return c_atten, generated_ids, ids


def colorize(attrs, cmap='bwr'):

    cmap_bound = tf.reduce_max(tf.abs(attrs))

    norm = Normalize(vmin=-cmap_bound, vmax=cmap_bound)

    cmap = mpl.cm.get_cmap(cmap)
    colors = list(map(lambda x: rgb2hex(cmap(norm(x))), attrs))

    return colors

def  hlstr(string, color='white'):

    return f"<mark style=background-color:{color}>{string} </mark>"



def color_(max_atten_per_ipword , input_tokens):
#     print("*********************")
    input_tokens = [x[1:] for x in input_tokens]
#     print("Tokens are : ", input_tokens)

#     print("*********************")
    colors = colorize(max_atten_per_ipword)
    colored_input=[]
    display(HTML("".join(list(map(hlstr, input_tokens, colors)))))


def cross_atten(model,tokenizer,parameters,sent, device):

    c_atten, generated_ids, input_ids = predict(model,tokenizer,parameters,sent, device)

    target_input_attn = get_max_attn(c_atten)

    max_atten_per_ipword = []
    for ipword in range(512):
        max_ = 0.0
        for target in range(len(target_input_attn)):
            if(max_ <= target_input_attn[target][ipword]):
                max_ = target_input_attn[target][ipword]
        max_atten_per_ipword.append(max_)
    input_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    input_tokens = [token for token in input_tokens if token != '<pad>']

    color_(max_atten_per_ipword , input_tokens)

def cross_atten_per_word(model,tokenizer,parameters,sent, device):

    c_atten, generated_ids, input_ids = predict_(model,tokenizer,parameters,sent, device)
    tarid = (int)(input("Enter the input id of the target word to be analysed :"))
    target_input_attn = get_max_attn(c_atten)

    input_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    input_tokens = [token for token in input_tokens if token != '<pad>']

    color_(target_input_attn[tarid] , input_tokens)


In [ ]:
# load the pre-trained best-checkpoint model
from datasets import load_dataset
def inference(model, device, tokenizer, model_params,i):
#     dataset = load_dataset("multi_news")
#     test = dataset["test"]
    #test = test.remove_columns(["id"])
    test= load_dataset('multi_news', split='test[:5%]')

    sent = test[i]["document"]  # taking an example sentence
    sent = "summarize: " + sent
    sent = " ".join(sent.split())

    source = tokenizer.__call__(
            [sent],
            max_length=model_params["MAX_SOURCE_TEXT_LENGTH"],
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    ids = source["input_ids"]
    mask = source["attention_mask"]

    model.eval()
    with torch.no_grad():
        ids = ids.to(device, dtype = torch.long)
        mask = mask.to(device, dtype = torch.long)

        generated_ids = model.generate(
              input_ids = ids,
              attention_mask = mask,
              max_length=150,
              #num_beams=2,
              repetition_penalty=2.5,  # there is a research paper for this
              #length_penalty=1.0,  # > 0 encourages to generate short sentences, < 0 to generate long sentences
              #early_stopping=True  # stops beam search when number of beams sentences are generated per batch
              )

        preds = tokenizer.decode(generated_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

        print("Input dialogue is: ", test[i]["summary"])
        print("###############################")
        print("Output summary is: ", preds)
        c=rouge_calc(preds,sent)
        print("###############################")
        cross_atten(model,tokenizer,model_params,test["document"][i], device)
        print(c)



cuda =  torch.cuda.is_available()
device = torch.device("cuda") if cuda else torch.device("cpu")


inference(model, device, tokenizer, model_params,35)